### Libraries


In [1]:
import requests  # for making http request to the API
from datetime import date, timedelta #for handling date and time
import pandas as pd  # for data transformation and processing

In [5]:
import requests  # Make HTTP requests to the API
import pandas as pd  # Transform and process data
from datetime import date, timedelta  # Handle dates and times

ModuleNotFoundError: No module named 'requests'

In [6]:
%pip install requests pandas

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached idna-3.18-py3-none-any.whl (65 kB)
Using cached urllib3-2.7.0-py3-none-any.whl (131 kB)
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ----- ---------------------------------- 1.3/9.8 MB 6.4 MB/s eta 0:00:02
   ---------- ----------------------------- 2.6/9.8 MB 6.4 MB/s eta 0:00:02
   ---------------- ----------------------- 3.9/9.8 MB 6.4 MB/s eta 0:00:01
   --------------------- ------------------ 5.2/9.8 MB 6.4 MB/s eta 0:00:01
   -------------------------- ------------- 6.6/9.8 MB 6.4 MB/s eta 0:00:01
   -------------------------------- ------- 7.9/9.8 MB 6.3 MB/s eta 0:00:01
   ------------------------------------- -- 9.2/9.8 MB 6.4 MB/


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import requests
import pandas as pd
from datetime import date, timedelta

print("Imports successful")

Imports successful


### Extraction

In [20]:
# set out target date
date_target = date.today()
URL = f"https://api.carbonintensity.org.uk/regional/intensity/{date_target}/pt24h" #set your url
#date_target

In [21]:
# set our api response
headers = {
    'Accept': 'application/json'
}

# make a request to their server
response = requests.get(URL, headers=headers)
print(response.status_code)
print(f"Gotten response from {URL} for {date_target}")

api_data = response.json()['data']

200
Gotten response from https://api.carbonintensity.org.uk/regional/intensity/2026-08-15/pt24h for 2026-08-15


In [22]:
api_data

[{'from': '2026-08-13T23:30Z',
  'to': '2026-08-14T00:00Z',
  'regions': [{'regionid': 1,
    'dnoregion': 'Scottish Hydro Electric Power Distribution',
    'shortname': 'North Scotland',
    'intensity': {'forecast': 0, 'index': 'very low'},
    'generationmix': [{'fuel': 'biomass', 'perc': 0},
     {'fuel': 'coal', 'perc': 0},
     {'fuel': 'imports', 'perc': 0},
     {'fuel': 'gas', 'perc': 0},
     {'fuel': 'nuclear', 'perc': 0},
     {'fuel': 'other', 'perc': 0},
     {'fuel': 'hydro', 'perc': 0},
     {'fuel': 'solar', 'perc': 0},
     {'fuel': 'wind', 'perc': 100}]},
   {'regionid': 2,
    'dnoregion': 'SP Distribution',
    'shortname': 'South Scotland',
    'intensity': {'forecast': 3, 'index': 'very low'},
    'generationmix': [{'fuel': 'biomass', 'perc': 2},
     {'fuel': 'coal', 'perc': 0},
     {'fuel': 'imports', 'perc': 5.2},
     {'fuel': 'gas', 'perc': 0},
     {'fuel': 'nuclear', 'perc': 50.1},
     {'fuel': 'other', 'perc': 0},
     {'fuel': 'hydro', 'perc': 0},
    

### Transformation

In [23]:
subset = api_data[0]
subset

{'from': '2026-08-13T23:30Z',
 'to': '2026-08-14T00:00Z',
 'regions': [{'regionid': 1,
   'dnoregion': 'Scottish Hydro Electric Power Distribution',
   'shortname': 'North Scotland',
   'intensity': {'forecast': 0, 'index': 'very low'},
   'generationmix': [{'fuel': 'biomass', 'perc': 0},
    {'fuel': 'coal', 'perc': 0},
    {'fuel': 'imports', 'perc': 0},
    {'fuel': 'gas', 'perc': 0},
    {'fuel': 'nuclear', 'perc': 0},
    {'fuel': 'other', 'perc': 0},
    {'fuel': 'hydro', 'perc': 0},
    {'fuel': 'solar', 'perc': 0},
    {'fuel': 'wind', 'perc': 100}]},
  {'regionid': 2,
   'dnoregion': 'SP Distribution',
   'shortname': 'South Scotland',
   'intensity': {'forecast': 3, 'index': 'very low'},
   'generationmix': [{'fuel': 'biomass', 'perc': 2},
    {'fuel': 'coal', 'perc': 0},
    {'fuel': 'imports', 'perc': 5.2},
    {'fuel': 'gas', 'perc': 0},
    {'fuel': 'nuclear', 'perc': 50.1},
    {'fuel': 'other', 'perc': 0},
    {'fuel': 'hydro', 'perc': 0},
    {'fuel': 'solar', 'perc': 

In [ ]:
# get a particular shortname
subset['regions'][0]['shortname']

'North Scotland'

In [25]:
# to get all the shortnames for the regions

for i in subset['regions']:
    print(i['shortname'])

North Scotland
South Scotland
North West England
North East England
Yorkshire
North Wales & Merseyside
South Wales
West Midlands
East Midlands
East England
South West England
South England
London
South East England
England
Scotland
Wales
GB


In [ ]:
# to get all the index of intensity for the regions

for i in subset['regions']:
    print(i['intensity']['index'])

very low
very low
moderate
very low
moderate
moderate
very high
high
very high
very high
very high
very high
very high
high
high
low
very high
high


: 

In [ ]:
# flatten the data for that dataframe creation
# for each of the intervals [30 mins] get me the regions
# for each of the regions get me the data
# we are going to create a dictionary for each region at the 30 minute mark

records = []
for interval in api_data:
    for region in interval['regions']:
        # create a dict for each region at the 30 min mark
        row = {
            'regionid': region['regionid'], 'shortname': region['shortname']
        }

In [ ]:
# Convert to a dataframe
df = pd.DataFrame(records)
print(f"successfully converted api_data to dataframe with {len(df)}")